In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

import itertools
from pybedtools import BedTool

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from scipy.stats import binomtest, ttest_ind

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
clusts = list(np.loadtxt(f'{projdir}/txt/allc_table/rna_cluster_donor.txt', dtype=str))

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_a{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.shape

In [ ]:
pmds.head()

In [ ]:
auto = [f'chr{x}' for x in range(1,23)]

In [ ]:
indir = '/u/project/cluo/terencew/reference/hg38_igvf'
genes = pd.read_csv(f'{indir}/gencode.v43.genes.gtf', sep='\t',  header=None)
genes.index = [f'{x}_{y}' for x,y in zip(genes[0], genes[3])]
genes = genes[~genes.index.duplicated()]
mask = [x in auto for x in genes[0]]
genes = genes[mask]
genes.shape

In [ ]:
mask = [x.split(';')[1].split('\"')[1] == 'protein_coding' for x in genes[8]]
pc_genes = genes[mask]
pc_genes.shape

In [ ]:
pc_genes.head()

In [ ]:
indir = '/u/project/cluo/terencew/reference/hg38_igvf/bed/genes/'
genes_bed = pd.read_csv(f'{indir}/gencode.v43.autosome.dedup.names.bed', sep='\t',  header=None)
genes_bed.index = [f'{x}_{y}' for x,y in zip(genes_bed[0], genes_bed[1])]
genes_bed = genes_bed[~genes_bed.index.duplicated()]
genes_bed.shape

In [ ]:
final_genes = genes_bed.reindex(pc_genes.index).dropna()
final_genes[1] = final_genes[1].astype(int)
final_genes[2] = final_genes[2].astype(int)
final_genes.shape

In [ ]:
final_genes.head()

In [ ]:
pmds['size'] = pmds[2] - pmds[1]
pmds['size'].mean()

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
genes_bt = BedTool.from_dataframe(final_genes)

genes_intersect = genes_bt.intersect(pmds_bt).to_dataframe()
genes_intersect.shape

In [ ]:
%%time

gfile = '/u/project/cluo/terencew/reference/hg38_igvf/GRCh38.autosomal.chromsizes'

np.random.seed(1)

n_iters = 1000
seeds = np.random.randint(1e6, size=n_iters)

def shuffle_pmds(s):
    shuffled_pmds = pmds_bt.shuffle(g=gfile, seed=s)
    count = genes_bt.intersect(shuffled_pmds, u=True).count()
    return count

with ProcessPoolExecutor(max_workers=8) as executor:
    gene_results = list(tqdm(executor.map(shuffle_pmds, seeds), total=len(seeds)))

In [ ]:
###
obs = genes_intersect.shape[0]
exp = np.mean(gene_results)
obs_exp = obs / exp

###
total_genes = genes.shape[0]
p = exp / total_genes
pval = binomtest(obs, n=total_genes, p=p, alternative="two-sided")
obs_exp, pval

In [ ]:
obs, exp

In [ ]:
obs

In [ ]:
obs_df = pd.DataFrame([obs])
obs_df['type'] = ['observed']
exp_df = pd.DataFrame(gene_results)
exp_df['type'] = 'expected'

In [ ]:
to_plot = pd.concat([obs_df, exp_df])
to_plot.columns = ['num', 'type']
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(6, 5))

x_lab = 'num'
y_lab = 'score'
hue = 'type'

ax = sns.histplot(to_plot, x=x_lab, hue=hue)
                 
ax.set_xlabel('# genes overlapping PMDs', fontsize=12)
ax.set_ylabel('', fontsize=12)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelrotation=45)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
to_plot.to_csv(f'{outdir}/pmds_genes_exp_overlap.csv', sep='\t')

### let's do repli-seq signal now

In [ ]:
indir = '/u/project/cluo/terencew/public_datasets/encode/stamato_fibro_repli'
repli = pd.read_csv(f'{indir}/bed/ENCFF001GII.hg38.bed', sep='\t', header=None)
repli.shape

In [ ]:
repli.head()

In [ ]:
sns.histplot(repli, x=3)

In [ ]:
# tmp_pmds.shape

In [ ]:
repli_bt = BedTool.from_dataframe(repli)
pmds_bt = BedTool.from_dataframe(pmds)
intersect = repli_bt.intersect(pmds_bt).to_dataframe()
comp = repli_bt.intersect(pmds_bt, v=True).to_dataframe()
intersect.shape, comp.shape

In [ ]:
intersect.head()

In [ ]:
comp.head()

In [ ]:
intersect['type'] = 'PMDs'
comp['type'] = 'Non-PMDs'
merged = pd.concat([intersect, comp])
merged.shape

In [ ]:
merged.head()

In [ ]:
merged['name'].max(), merged['name'].min()

In [ ]:
ttest_ind(intersect['name'], comp['name'])

In [ ]:
fig, axes = plt.subplots(1, figsize=(6, 5))

x_lab = 'type'
y_lab = 'name'

ax = sns.boxplot(merged, x=x_lab, y=y_lab, showfliers=False)
                 
ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Repli-seq signal', fontsize=12)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelrotation=45)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
merged.to_csv(f'{outdir}/pmds_repli_seq.csv', sep='\t')

In [ ]:
!date